In [1]:
import numpy as np 
import tensorflow as tf
import keras 
from sklearn.model_selection import train_test_split
import librosa

import matplotlib.pyplot as plt
import sys
sys.path.append("/home/ciona/projects/RCOLM/data_models/MyModels")  
from mymodels import MyModels

2025-11-17 13:42:09.291154: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-17 13:42:09.535007: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-17 13:42:10.825052: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


##  Main funkction

In [2]:
class MUSDB_augmentation:
    def change_volume(x, min_gain, max_gain):
        db = np.random.uniform(min_gain, max_gain)
        factor  = 10 ** (db/20)
        return x * factor
    
    def noise(x, noise_level):
        noise = np.random.randn(*x.shape)*nosie_level
        return x + noise

    
    def time_stretch(x, rate=1.1):
        return librosa.effects.time_stretch(x, rate)


    def bandpass(x, low, high, sr=44100):
        b,a = butter(4, [low/(sr/2), high/(sr/2)], btype='band')
        return lfilter(b, a, x)

    def train():
        print('dupa')

In [3]:
class Train:
    @staticmethod 
    def create_dataset(X, y, batch_size=2):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)

        print("Creating dataset from X.shape =", X.shape, " y.shape =", y.shape, flush=True)

        ds = tf.data.Dataset.from_tensor_slices((X, y))
        ds = ds.shuffle(buffer_size=min(64, len(X)))
        ds = ds.batch(batch_size)
        ds = ds.prefetch(tf.data.AUTOTUNE)
        return ds


    @staticmethod
    def training():
        DATA_PATH = "/home/ciona/projects/RCOLM/data_models/training/musdb18/dataset/dataset.npz"
        data = np.load(DATA_PATH)
        X = data["X"]
        y = data["Y"]
        #normalization
        
        print("X min:", np.min(X))
        print("X max:", np.max(X))
        print("X mean:", np.mean(X))
        print("Is NaN:", np.isnan(X).any())
        print("Is inf:", np.isinf(X).any())

        print("\ny min:", np.min(y))
        print("y max:", np.max(y))
        print("y mean:", np.mean(y))
        print("Is NaN:", np.isnan(y).any())
        print("Is inf:", np.isinf(y).any())


        print("X shape:", X.shape)
        print("y shape:", y.shape)
        
        X = (X+80.0)/80.0
        y = (y+80)/80   
# augmentation


# 🔹 10% danych
        ratio = 0.1
        n = X.shape[0]
        n_subset = int(n * ratio)

        idx = np.random.permutation(n)[:n_subset]  # losowe indeksy
        X = X[idx]
        y = y[idx]

        print("Dataset fragment:")
        print("X shape:", X.shape)
        print("y shape:", y.shape)        

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

        # Wejściowy kształt (tu U-Net na pełnym rozmiarze 360x216)
        H, W, C_in = X.shape[1], X.shape[2], X.shape[3]
        C_out = y.shape[3]   # 4

        input_shape = (H, W, C_in)   # (360, 216, 1)
        input_bins = C_out           # 4

        # Uważaj: to są duże macierze – batch_size koniecznie mały
        train_ds = Train.create_dataset(X_train, y_train, batch_size=1)
        test_ds  = Train.create_dataset(X_test,  y_test,  batch_size=1)

        print("Dataset ready, compiling model...")

        model = MyModels.build_unet(input_shape, num_bins=input_bins)
        model.compile(optimizer='adam', loss='mse', metrics = 
        [tf.keras.metrics.MeanSquaredError(name="mse"),
        tf.keras.metrics.MeanAbsoluteError(name="mae"),])

        print("Model compiled, starting fit...")

        history = model.fit(
            train_ds,
            epochs=10,
            verbose=1,
            validation_data=test_ds,
        )





        return model, history

In [4]:
Train.training()

X min: -80.0
X max: 0.0
X mean: -35.76687
Is NaN: False
Is inf: False

y min: -80.0
y max: 0.0
y mean: -41.90101
Is NaN: False
Is inf: False
X shape: (6999, 360, 216, 1)
y shape: (6999, 360, 216, 4)
Dataset fragment:
X shape: (699, 360, 216, 1)
y shape: (699, 360, 216, 4)
Creating dataset from X.shape = (489, 360, 216, 1)  y.shape = (489, 360, 216, 4)


I0000 00:00:1763383405.610117  122737 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6047 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Creating dataset from X.shape = (210, 360, 216, 1)  y.shape = (210, 360, 216, 4)
Dataset ready, compiling model...
Model compiled, starting fit...
Epoch 1/10


2025-11-17 13:43:30.811075: I external/local_xla/xla/service/service.cc:163] XLA service 0x782fc0003dd0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-17 13:43:30.811088: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 Laptop GPU, Compute Capability 8.9
2025-11-17 13:43:30.942447: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-17 13:43:31.654145: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91600
2025-11-17 13:43:32.760076: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 6.16GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.
2025-11-17 13:43:32.81097

 10/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.1302 - mae: 0.2948 - mse: 0.1302

I0000 00:00:1763383417.261148  122870 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


489/489 ━━━━━━━━━━━━━━━━━━━━ 19s 19ms/step - loss: 0.0629 - mae: 0.1909 - mse: 0.0629 - val_loss: 0.0688 - val_mae: 0.2015 - val_mse: 0.0688
Epoch 2/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0596 - mae: 0.1834 - mse: 0.0596 - val_loss: 0.0714 - val_mae: 0.2012 - val_mse: 0.0714
Epoch 3/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0589 - mae: 0.1820 - mse: 0.0589 - val_loss: 0.0680 - val_mae: 0.1903 - val_mse: 0.0680
Epoch 4/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0581 - mae: 0.1806 - mse: 0.0581 - val_loss: 0.0652 - val_mae: 0.1853 - val_mse: 0.0652
Epoch 5/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0575 - mae: 0.1791 - mse: 0.0575 - val_loss: 0.0689 - val_mae: 0.1915 - val_mse: 0.0689
Epoch 6/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0569 - mae: 0.1782 - mse: 0.0569 - val_loss: 0.0681 - val_mae: 0.1909 - val_mse: 0.0681
Epoch 7/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 8s 17ms/step - loss: 0.0573 - mae: 0.1795 - mse: 0.0573 - val_lo

(<Functional name=Unet_multiF0, built=True>,
 <keras.src.callbacks.history.History at 0x78310517e590>)

In [6]:
# opis modelu oraz epoch przejsciowy
input_bins = 5
input_shape = (256, 256, 1)

model = MyModels.build_unet(input_shape, num_bins = input_bins)
model.summary()
# 4. Testowy forward pass na losowym tensorze
x_test = tf.random.normal((2, *input_shape))  # batch_size = 2
y_pred = model(x_test)
print("Wejście:", x_test.shape)
print("Wyjście:", y_pred.shape)

# 5. Test treningu – fejkowe dane
y_fake = tf.random.uniform(y_pred.shape)    

model.compile(optimizer='adam', loss='binary_crossentropy')

history = model.fit(
    x_test,
    y_fake,
    epochs=1,
    batch_size=2,
    verbose=1
)

Model: "Unet_multiF0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 256, 256,  │        320 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_18 (ReLU)     │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 256, 256,  │      9,248 │ re_lu_18[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 256,  │        128 │ conv2d_20[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_19 (ReLU)     │ (None, 256, 256,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 128, 128,  │          0 │ re_lu_19[0][0]    │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 128, 128,  │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_21[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_20 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 128, 128,  │     36,928 │ re_lu_20[0][0]    │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        256 │ conv2d_22[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_21 (ReLU)     │ (None, 128, 128,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 64, 64,    │          0 │ re_lu_21[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_5[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        512 │ conv2d_23[0][0] 

 Total params: 6,007,693 (22.92 MB)

 Trainable params: 6,002,397 (22.90 MB)

 Non-trainable params: 5,296 (20.69 KB)

Wejście: (2, 256, 256, 1)
Wyjście: (2, 256, 256, 5)
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - loss: 0.7614
